# SeaFour Retrieval Engine Submission Notebook

This notebook is a cleaned, reproducible retrieval pipeline for the competition. It keeps the original project logic, but focuses on the final workflow we actually use: shared preprocessing, cached retrieval artifacts, leakage-safe evaluation, and Kaggle submission generation.

Implemented retrievers:
- **TF-IDF** for a lightweight lexical baseline
- **BM25+** for a stronger lexical baseline
- **Embedding retrieval** for the active dense first-stage method

If a fresh environment is missing packages, install them before running the notebook:

```python
%pip install -q rank_bm25 sentence-transformers scikit-learn
```


## Retrieval Engine Overview

The active notebook flow is:

1. Load documents, labeled train queries, unlabeled Kaggle test queries, and ground truth.
2. Build shared normalized `content` fields for documents and queries.
3. Split labeled queries stratified into train / validation / hold-out test (`60/30/10`).
4. Fit learned query-side preprocessing only on the split-train data.
5. Build cached document-side retrieval artifacts.
6. Train the cross-encoder only on split-train queries.
7. Select on validation, check once on hold-out test, then generate production predictions.

The document corpus is fixed for retrieval, while the leakage-sensitive supervised components only see the split-train labeled queries.


## Retriever Choice

The notebook keeps TF-IDF and BM25+ as lexical baselines, but the active default path uses embeddings because this dataset has frequent wording mismatch between queries and relevant documents.

Practical guide:
- use **TF-IDF** when you want the simplest fast baseline
- use **BM25+** when exact wording matters and document lengths vary
- use **embeddings** when recall under paraphrasing matters most

`top_k` controls the first-stage candidate pool:
- smaller `top_k` helps precision and latency
- larger `top_k` helps recall but increases runtime and noise

In the current notebook, `top_k=7500` is the active retrieval depth for evaluation and submission.


In [ ]:
# Imports, paths, and experiment configuration.
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, Sequence, TypedDict
import csv
import hashlib
import json
import os
import pickle
import re
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 5)

ModelName = Literal["tfidf", "bm25", "embedding"]


class RetrievalResult(TypedDict):
    """One ranked retrieval result for a single query."""

    query_id: str
    relevant_docs: list[str]


class GroundTruthEntry(TypedDict):
    """Ground-truth annotations for a single training query."""

    relevant_doc_ids: set[str]
    total_relevant_docs: int
    category: str | None


@dataclass(frozen=True)
class RuntimePaths:
    """Resolved filesystem locations for the active notebook runtime."""

    runtime_env: Literal["colab", "kaggle", "local"]
    project_dir: Path
    work_dir: Path
    data_dir: Path
    cache_dir: Path
    output_path: Path


def detect_runtime_environment() -> str:
    from pathlib import Path
    import os

    if Path("/kaggle/input").exists() or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"

    try:
        import google.colab  # type: ignore  # noqa: F401

        return "colab"
    except Exception:
        return "local"


def find_colab_project_dir(project_name: str = "retrieval_project") -> Path | None:
    """Locate the project folder on Google Drive when running in Colab."""
    drive_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    for drive_root in drive_candidates:
        if not drive_root.exists():
            continue

        direct_candidate = drive_root / project_name
        if (direct_candidate / "data" / "docs.json").exists():
            return direct_candidate

        for candidate in drive_root.rglob(project_name):
            if candidate.is_dir() and (candidate / "data" / "docs.json").exists():
                return candidate

    return None


def find_local_project_data_dir() -> Path | None:
    """Find the `data/` directory near the current working directory."""
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        data_dir = root / "data"
        if (data_dir / "docs.json").exists():
            return data_dir
    return None


def resolve_runtime_paths(output_filename: str = "solutions_SeaFour.csv") -> RuntimePaths:
    """Resolve project, data, cache, and output paths for the active runtime."""
    runtime_env = detect_runtime_environment()
    project_dir: Path | None = None
    work_dir = Path.cwd()

    if runtime_env == "colab":
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        project_dir = find_colab_project_dir("retrieval_project")
        if project_dir is None:
            raise FileNotFoundError(
                "Google Drive is mounted, but `retrieval_project/data/docs.json` was not found."
            )
        os.chdir(project_dir)
        work_dir = project_dir
        data_dir = project_dir / "data"
    elif runtime_env == "kaggle":
        data_dir = Path("/kaggle/input/competitions/retrieval-engine-competition")
        if not data_dir.exists():
            raise FileNotFoundError(f"Kaggle data directory not found: {data_dir}")
        project_dir = Path.cwd()
    else:
        data_dir = find_local_project_data_dir()
        if data_dir is None:
            raise FileNotFoundError(
                "Could not find `data/docs.json` near the current working directory."
            )
        project_dir = data_dir.parent
        work_dir = project_dir
        os.chdir(work_dir)

    cache_dir = work_dir / "cache"
    output_path = work_dir / output_filename
    return RuntimePaths(
        runtime_env=runtime_env,
        project_dir=project_dir or work_dir,
        work_dir=work_dir,
        data_dir=data_dir,
        cache_dir=cache_dir,
        output_path=output_path,
    )


FINAL_MODEL: ModelName = "embedding"
EVALUATION_MODELS: tuple[ModelName, ...] = ("embedding",)
EVALUATION_TOP_KS: tuple[int, ...] = (7_500,)
SUBMIT_TOP_K = 7_500

TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.30
TEST_FRACTION = 0.10
SPLIT_RANDOM_SEED = 42

ENABLE_CATEGORY_BOOST = True
CATEGORY_MATCH_BONUS = 2.0
ENABLE_CROSS_ENCODER_RERANK = True
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
CROSS_ENCODER_EPOCHS = 5
CROSS_ENCODER_BATCH_SIZE = 32
CROSS_ENCODER_MAX_LENGTH = 256
CROSS_ENCODER_MAX_POSITIVES_PER_QUERY = 4
CROSS_ENCODER_NEGATIVES_PER_POSITIVE = 4
CROSS_ENCODER_TRAIN_QUERY_LIMIT = 327
CROSS_ENCODER_HARD_NEGATIVE_TOP_K = 200
CROSS_ENCODER_RANDOM_SEED = 42
CROSS_ENCODER_RERANK_TOP_M = 150
CROSS_ENCODER_INFER_BATCH_SIZE = 64
CROSS_ENCODER_FP16 = True
ENABLE_CROSS_ENCODER_CACHE = True

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 256
EMBEDDING_QUERY_CHUNK_SIZE = 32

DOCUMENT_TEXT_COLUMNS: tuple[str, ...] = ("title", "text", "tags")
RETRIEVAL_QUERY_COLUMNS: tuple[str, ...] = ("title", "text")
CLASSIFIER_USE_QUERY_TAGS = True

NORMALIZATION_CONFIG = {
    "lowercase": True,
    "replace_separators": True,
    "separator_chars": "-_/",
    "collapse_whitespace": True,
    "strip": True,
}
TOKEN_PATTERN = r"[a-z0-9]+"
STOPWORD_FILTER_ENABLED = True
STOPWORD_LANGUAGE = "english"
CUSTOM_STOPWORDS: set[str] = set()

TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
    "stop_words": STOPWORD_LANGUAGE if STOPWORD_FILTER_ENABLED else None,
}
CLASSIFIER_TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
    "stop_words": STOPWORD_LANGUAGE if STOPWORD_FILTER_ENABLED else None,
}
BM25_CONFIG = {
    "k1": 1.5,
    "b": 0.75,
    "delta": 1.0,
}

ENABLE_EMBEDDING_CACHE = True
ENABLE_CLASSIC_CACHE = True

if not np.isclose(TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION, 1.0):
    raise ValueError("TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION must equal 1.0")

PATHS = resolve_runtime_paths()
MODEL_CACHE_DIR = PATHS.cache_dir / "sentence_transformers"
EMBEDDING_CACHE_DIR = PATHS.cache_dir / "embeddings"
TFIDF_CACHE_DIR = PATHS.cache_dir / "tfidf"
BM25_CACHE_DIR = PATHS.cache_dir / "bm25"
CLASSIFIER_CACHE_DIR = PATHS.cache_dir / "classifier"
CROSS_ENCODER_CACHE_DIR = PATHS.cache_dir / "cross_encoder"

for cache_path in [
    PATHS.cache_dir,
    MODEL_CACHE_DIR,
    EMBEDDING_CACHE_DIR,
    TFIDF_CACHE_DIR,
    BM25_CACHE_DIR,
    CLASSIFIER_CACHE_DIR,
    CROSS_ENCODER_CACHE_DIR,
]:
    cache_path.mkdir(parents=True, exist_ok=True)

print(f"Runtime environment: {PATHS.runtime_env}")
print(f"Project directory  : {PATHS.project_dir}")
print(f"Working directory  : {PATHS.work_dir}")
print(f"Data directory     : {PATHS.data_dir}")
print(f"Output path        : {PATHS.output_path}")


## Data Loading

This section loads the raw competition files and validates the expected schema early. The checks are intentionally strict so notebook failures happen close to the source of the problem instead of later in the retrieval pipeline.


In [ ]:
def require_columns(frame: pd.DataFrame, required_columns: Sequence[str], frame_name: str) -> None:
    """Raise a clear error if a dataframe is missing required columns."""
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{frame_name} is missing required columns: {missing_columns}")


def ensure_unique_ids(frame: pd.DataFrame, frame_name: str) -> None:
    """Ensure that the `id` column exists and does not contain duplicates."""
    require_columns(frame, ["id"], frame_name)
    if not frame["id"].astype(str).is_unique:
        raise ValueError(f"{frame_name} contains duplicate ids, which would break retrieval output mapping.")


def load_json_frame(path: Path, frame_name: str) -> pd.DataFrame:
    """Load one JSON competition file into a dataframe."""
    if not path.exists():
        raise FileNotFoundError(f"{frame_name} file not found: {path}")
    return pd.read_json(path)


docs_raw_df = load_json_frame(PATHS.data_dir / "docs.json", "Documents")
train_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_train.json", "Train queries")
test_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_test.json", "Test queries")
sample_submission_path = PATHS.data_dir / "submission.csv"
ground_truth_path = PATHS.data_dir / "qgts_train.json"

sample_submission_df = pd.read_csv(sample_submission_path)

require_columns(docs_raw_df, ["id", "title", "text", "tags", "category"], "Documents")
require_columns(train_queries_raw_df, ["id", "title", "text", "tags", "category"], "Train queries")
require_columns(test_queries_raw_df, ["id", "title", "text", "tags"], "Test queries")
require_columns(sample_submission_df, ["query_id", "relevant_doc_ids", "category"], "Sample submission")

ensure_unique_ids(docs_raw_df, "Documents")
ensure_unique_ids(train_queries_raw_df, "Train queries")
ensure_unique_ids(test_queries_raw_df, "Test queries")

print(f"Documents      : {len(docs_raw_df):,}")
print(f"Train queries  : {len(train_queries_raw_df):,}")
print(f"Test queries   : {len(test_queries_raw_df):,}")
print(f"Sample rows    : {len(sample_submission_df):,}")


## Preprocessing

This notebook now separates **fit-free normalization** from **learned preprocessing**:

- fit-free normalization: text cleaning, separator replacement, lowercasing, whitespace cleanup, and shared `content` construction
- learned preprocessing: the TF-IDF vectorizer used inside the lightweight query category classifier

The normalization rules are shared across the whole notebook, but the fitted classifier vectorizer is learned only on the 60% split-train queries and then reused with `.transform(...)` on validation, hold-out test, and final production queries. That is the key leakage-prevention change.


In [ ]:
_TOKEN_RE = re.compile(TOKEN_PATTERN)
_FALLBACK_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "if", "in", "into",
    "is", "it", "no", "not", "of", "on", "or", "such", "that", "the", "their", "then", "there",
    "these", "they", "this", "to", "was", "will", "with",
}
if STOPWORD_FILTER_ENABLED and STOPWORD_LANGUAGE != "english":
    raise ValueError("Only STOPWORD_LANGUAGE='english' is supported in this notebook.")
if STOPWORD_FILTER_ENABLED:
    try:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        STOPWORDS = set(ENGLISH_STOP_WORDS)
    except Exception:
        STOPWORDS = set(_FALLBACK_STOPWORDS)
    STOPWORDS = STOPWORDS.union({word.lower() for word in CUSTOM_STOPWORDS})
else:
    STOPWORDS = set()
_WHITESPACE_RE = re.compile(r"\s+")
_SEPARATOR_RE = re.compile(f"[{re.escape(NORMALIZATION_CONFIG['separator_chars'])}]")


def value_to_text(value: Any) -> str:
    """Convert raw dataframe values into plain text.

    Args:
        value: A scalar, list-like, or missing value from a dataframe cell.

    Returns:
        A string representation suitable for text normalization.
    """
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(text: Any) -> str:
    """Apply the shared normalization policy used across retrievers.

    Args:
        text: Raw text or text-like content.

    Returns:
        Normalized text with separators replaced, whitespace collapsed, and casing standardized.
    """
    if text is None:
        cleaned_text = ""
    elif not isinstance(text, str) and pd.isna(text):
        cleaned_text = ""
    else:
        cleaned_text = str(text)

    if NORMALIZATION_CONFIG["replace_separators"]:
        cleaned_text = _SEPARATOR_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["lowercase"]:
        cleaned_text = cleaned_text.lower()
    if NORMALIZATION_CONFIG["collapse_whitespace"]:
        cleaned_text = _WHITESPACE_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["strip"]:
        cleaned_text = cleaned_text.strip()
    return cleaned_text


def build_content_frame(frame: pd.DataFrame, text_columns: Sequence[str]) -> pd.DataFrame:
    """Build a dataframe with a normalized `content` column.

    Args:
        frame: Input dataframe that must contain `id` and the requested text columns.
        text_columns: Columns to concatenate into the normalized content field.

    Returns:
        A copy of the input dataframe with string ids and a new `content` column.
    """
    require_columns(frame, ["id"], "Input frame")
    output_frame = frame.copy()
    text_parts: list[list[str]] = []

    for column in text_columns:
        if column in output_frame.columns:
            text_parts.append(output_frame[column].map(value_to_text).tolist())
        else:
            text_parts.append([""] * len(output_frame))

    merged_text = [" ".join(parts) for parts in zip(*text_parts)]
    output_frame["content"] = [normalize_text(text) for text in merged_text]
    output_frame["id"] = output_frame["id"].astype(str)
    return output_frame


def tokenize(text: str) -> list[str]:
    """Tokenize normalized text for lexical retrieval."""
    tokens = _TOKEN_RE.findall(normalize_text(text))
    if not STOPWORDS:
        return tokens
    return [token for token in tokens if token not in STOPWORDS]


def build_query_classifier_frame(query_frame: pd.DataFrame, include_tags: bool = CLASSIFIER_USE_QUERY_TAGS) -> pd.DataFrame:
    """Build the text view used by the category classifier.

    Args:
        query_frame: Raw query dataframe.
        include_tags: Whether to include the `tags` column when available.

    Returns:
        A dataframe with a classifier-oriented `content` column.
    """
    columns = ["title", "text"]
    if include_tags and "tags" in query_frame.columns:
        columns.append("tags")
    return build_content_frame(query_frame, columns)


docs_df = build_content_frame(docs_raw_df, DOCUMENT_TEXT_COLUMNS)
train_queries_df = build_content_frame(train_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)
test_queries_df = build_content_frame(test_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)

train_queries_classifier_df = build_query_classifier_frame(train_queries_raw_df)
test_queries_classifier_df = build_query_classifier_frame(test_queries_raw_df)

if docs_df["content"].eq("").all():
    raise ValueError("All document content is empty after preprocessing. Check the source columns or normalization.")

print(f"Average document length (chars): {docs_df['content'].str.len().mean():.1f}")
print(f"Average train query length      : {train_queries_df['content'].str.len().mean():.1f}")
print(f"Average test query length       : {test_queries_df['content'].str.len().mean():.1f}")


## Retrieval Index Construction

The main engineering goal in this section is to make expensive work reusable without accidentally refitting on evaluation data.

- document-side artifacts such as embeddings, TF-IDF indexes, and BM25 indexes are cached on the fixed corpus
- query-side fitted preprocessing is cached with a split-specific fingerprint, so the classifier learned on split-train data is reused exactly as-is on validation, hold-out test, and production queries

The dense retriever is still the most expensive component, so caching and chunked scoring matter most there.


In [ ]:
_MODEL_MEMORY_CACHE: dict[str, Any] = {}
_ARRAY_MEMORY_CACHE: dict[str, np.ndarray] = {}
_OBJECT_MEMORY_CACHE: dict[str, Any] = {}


def _safe_component(value: Any) -> str:
    """Make a string safe for use in cache file names."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))


def _hash_payload(payload: dict[str, Any]) -> str:
    """Create a short deterministic hash for cache keys."""
    raw_payload = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha1(raw_payload.encode("utf-8")).hexdigest()[:16]


def _normalization_signature() -> str:
    """Fingerprint the preprocessing configuration used by the retrievers."""
    payload = {
        "normalization": NORMALIZATION_CONFIG,
        "token_pattern": TOKEN_PATTERN,
        "stopword_filter_enabled": STOPWORD_FILTER_ENABLED,
        "stopword_language": STOPWORD_LANGUAGE,
        "stopword_hash": hashlib.sha1("\n".join(sorted(STOPWORDS)).encode("utf-8")).hexdigest()[:16],
    }
    return _hash_payload(payload)


def _dataframe_fingerprint(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    """Fingerprint selected dataframe columns for cache invalidation."""
    hasher = hashlib.sha1()
    hasher.update(str(len(frame)).encode("utf-8"))
    for column in columns:
        hasher.update(column.encode("utf-8"))
        column_hash = pd.util.hash_pandas_object(frame[column].astype(str), index=False).values
        hasher.update(column_hash.tobytes())
    return hasher.hexdigest()[:16]


def _load_pickle(path: Path) -> Any:
    """Load a pickled artifact from disk."""
    with open(path, "rb") as handle:
        return pickle.load(handle)


def _save_pickle(path: Path, artifact: Any) -> None:
    """Persist an artifact to disk with the highest pickle protocol."""
    with open(path, "wb") as handle:
        pickle.dump(artifact, handle, protocol=pickle.HIGHEST_PROTOCOL)


def _load_sentence_model(model_name: str) -> Any:
    """Load a Sentence-Transformer model, preferring the local cache when available."""
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    if model_name in _MODEL_MEMORY_CACHE:
        return _MODEL_MEMORY_CACHE[model_name]

    safe_model_name = _safe_component(model_name)
    local_model_dir = MODEL_CACHE_DIR / safe_model_name
    if local_model_dir.exists():
        print(f"Loading model weights from cache: {local_model_dir}")
        model = SentenceTransformer(str(local_model_dir))
    else:
        print(f"Downloading model weights: {model_name}")
        model = SentenceTransformer(model_name)
        local_model_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(local_model_dir))
        print(f"Saved model weights to cache: {local_model_dir}")

    _MODEL_MEMORY_CACHE[model_name] = model
    return model


def _load_or_encode_embeddings(
    frame: pd.DataFrame,
    kind: str,
    model: Any,
    model_name: str,
    batch_size: int,
) -> np.ndarray:
    """Load cached embeddings or encode them once and persist the result.

    Args:
        frame: Dataframe containing `id` and normalized `content`.
        kind: Human-readable cache prefix such as `docs` or `queries_train`.
        model: Loaded Sentence-Transformer model.
        model_name: Model identifier used in cache keys.
        batch_size: Sentence-Transformer encoding batch size.

    Returns:
        A float32 matrix of L2-normalized embeddings.
    """
    signature = _dataframe_fingerprint(frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_name = f"{kind}_{_safe_component(model_name)}_{normalization_signature}_{signature}.npy"
    cache_path = EMBEDDING_CACHE_DIR / cache_name
    memory_key = str(cache_path.resolve())

    if memory_key in _ARRAY_MEMORY_CACHE:
        return _ARRAY_MEMORY_CACHE[memory_key]

    if ENABLE_EMBEDDING_CACHE and cache_path.exists():
        print(f"Loading {kind} embeddings from cache: {cache_path.name}")
        embeddings = np.load(cache_path)
    else:
        print(f"Encoding {len(frame):,} {kind} rows...")
        embeddings = model.encode(
            frame["content"].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if ENABLE_EMBEDDING_CACHE:
            np.save(cache_path, embeddings)
            print(f"Saved {kind} embeddings to cache: {cache_path.name}")

    _ARRAY_MEMORY_CACHE[memory_key] = embeddings
    return embeddings


def _tfidf_param_candidates() -> list[dict[str, Any]]:
    """Return TF-IDF parameter settings including a safe fallback for very small corpora."""
    candidates = [dict(TFIDF_CONFIG)]
    min_df = TFIDF_CONFIG.get("min_df", 1)
    if isinstance(min_df, int) and min_df > 1:
        fallback = dict(TFIDF_CONFIG)
        fallback["min_df"] = 1
        candidates.append(fallback)
    return candidates


def build_or_load_tfidf_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF vectorizer and document matrix."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    doc_ids = docs_frame["id"].to_numpy()

    for params in _tfidf_param_candidates():
        cache_key = _hash_payload(
            {
                "docs_signature": docs_signature,
                "normalization_signature": normalization_signature,
                "tfidf_params": params,
            }
        )
        cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
        memory_key = str(cache_path.resolve())

        if memory_key in _OBJECT_MEMORY_CACHE:
            return _OBJECT_MEMORY_CACHE[memory_key]
        if ENABLE_CLASSIC_CACHE and cache_path.exists():
            print(f"Loading TF-IDF artifacts from cache: {cache_path.name}")
            artifacts = _load_pickle(cache_path)
            _OBJECT_MEMORY_CACHE[memory_key] = artifacts
            return artifacts

    params = dict(TFIDF_CONFIG)
    vectorizer = TfidfVectorizer(**params)
    try:
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])
    except ValueError as err:
        if "After pruning, no terms remain" not in str(err) or params.get("min_df", 1) == 1:
            raise
        params["min_df"] = 1
        vectorizer = TfidfVectorizer(**params)
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])

    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "tfidf_params": params,
        }
    )
    cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    artifacts = {
        "vectorizer": vectorizer,
        "doc_vectors": doc_vectors,
        "doc_ids": doc_ids,
        "params": params,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved TF-IDF artifacts to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_bm25_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached BM25+ index."""
    try:
        from rank_bm25 import BM25Plus
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `rank_bm25`. Install it with `%pip install rank_bm25`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "bm25_params": BM25_CONFIG,
        }
    )
    cache_path = BM25_CACHE_DIR / f"bm25_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading BM25 index from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    tokenized_corpus = [tokenize(text) for text in docs_frame["content"]]
    bm25 = BM25Plus(tokenized_corpus, **BM25_CONFIG)
    artifacts = {
        "bm25": bm25,
        "doc_ids": docs_frame["id"].to_numpy(),
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved BM25 index to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_category_classifier(train_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF + LinearSVC category classifier fit on the provided training frame."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    require_columns(train_frame, ["id", "content", "category"], "Classifier training data")
    train_signature = _dataframe_fingerprint(train_frame, ["id", "content", "category"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "train_signature": train_signature,
            "normalization_signature": normalization_signature,
            "classifier_tfidf_params": CLASSIFIER_TFIDF_CONFIG,
        }
    )
    cache_path = CLASSIFIER_CACHE_DIR / f"category_classifier_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading category classifier from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    vectorizer = TfidfVectorizer(**CLASSIFIER_TFIDF_CONFIG)
    train_vectors = vectorizer.fit_transform(train_frame["content"])
    classifier = LinearSVC()
    classifier.fit(train_vectors, train_frame["category"].astype(str))

    artifacts = {
        "vectorizer": vectorizer,
        "classifier": classifier,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved category classifier to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


## Retrieval Functions

The retrieval code is intentionally separated from index construction. That keeps the notebook easier to reason about and makes it straightforward to compare methods fairly.

The main performance changes in this refactor are:
- use **partial top-k selection** with `np.argpartition` instead of sorting every score vector fully
- score embedding queries in **chunks** to avoid creating unnecessarily large dense matrices
- reuse the **same max-k ranking** during offline sweeps and truncate it for smaller K values


In [ ]:
def validate_pipeline_settings(document_count: int) -> None:
    """Validate global retrieval and submission settings."""
    if FINAL_MODEL not in {"tfidf", "bm25", "embedding"}:
        raise ValueError(f"Unknown FINAL_MODEL: {FINAL_MODEL}")
    if any(model_name not in {"tfidf", "bm25", "embedding"} for model_name in EVALUATION_MODELS):
        raise ValueError(f"Unknown model in EVALUATION_MODELS: {EVALUATION_MODELS}")
    if document_count <= 0:
        raise ValueError("The document collection is empty.")
    if SUBMIT_TOP_K <= 0:
        raise ValueError("SUBMIT_TOP_K must be positive.")


def top_k_indices(score_vector: np.ndarray, top_k: int) -> np.ndarray:
    """Return indices of the top-k scores in descending order.

    This uses `np.argpartition` to avoid a full sort when only the largest values are needed.
    """
    if top_k <= 0:
        raise ValueError("top_k must be positive.")

    capped_top_k = min(top_k, score_vector.shape[0])
    if capped_top_k == score_vector.shape[0]:
        return np.argsort(score_vector)[::-1]

    candidate_indices = np.argpartition(score_vector, -capped_top_k)[-capped_top_k:]
    sorted_candidates = candidate_indices[np.argsort(score_vector[candidate_indices])[::-1]]
    return sorted_candidates


def truncate_results(results: list[RetrievalResult], top_k: int) -> list[RetrievalResult]:
    """Truncate a ranked result list to a smaller K without recomputing scores."""
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    return [
        {
            "query_id": result["query_id"],
            "relevant_docs": result["relevant_docs"][:top_k],
        }
        for result in results
    ]


def progress_interval(total_items: int, target_updates: int = 5) -> int:
    """Choose a lightweight logging interval for progress messages."""
    return max(1, total_items // max(1, target_updates))


def prepare_retriever(model_name: ModelName, docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Prepare and cache the artifacts required by one retriever."""
    if model_name == "tfidf":
        return build_or_load_tfidf_index(docs_frame)
    if model_name == "bm25":
        return build_or_load_bm25_index(docs_frame)
    if model_name == "embedding":
        model = _load_sentence_model(EMBEDDING_MODEL_NAME)
        doc_embeddings = _load_or_encode_embeddings(
            docs_frame,
            kind="docs",
            model=model,
            model_name=EMBEDDING_MODEL_NAME,
            batch_size=EMBEDDING_BATCH_SIZE,
        )
        return {
            "model": model,
            "doc_embeddings": doc_embeddings,
            "doc_ids": docs_frame["id"].to_numpy(),
        }
    raise ValueError(f"Unknown model: {model_name}")


def run_tfidf_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run sparse lexical retrieval with TF-IDF cosine similarity."""
    artifacts = prepared_artifacts or build_or_load_tfidf_index(docs_frame)
    vectorizer = artifacts["vectorizer"]
    doc_vectors = artifacts["doc_vectors"]
    doc_ids = artifacts["doc_ids"]
    query_vectors = vectorizer.transform(queries_frame["content"])
    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    log_every = progress_interval(len(query_ids))

    print(
        f"  [TF-IDF] vectorized {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, query_id in enumerate(query_ids):
        score_row = query_vectors[row_index] @ doc_vectors.T
        score_vector = np.asarray(score_row.toarray()).ravel()
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": query_id,
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_ids) - 1:
            print(f"  [TF-IDF] processed {row_index + 1:,}/{len(query_ids):,} queries")
    return results


def run_bm25_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run lexical retrieval with BM25+ over tokenized text."""
    artifacts = prepared_artifacts or build_or_load_bm25_index(docs_frame)
    bm25 = artifacts["bm25"]
    doc_ids = artifacts["doc_ids"]
    capped_top_k = min(top_k, len(doc_ids))
    query_pairs = list(queries_frame[["id", "content"]].itertuples(index=False, name=None))
    log_every = progress_interval(len(query_pairs))

    print(
        f"  [BM25+] scoring {len(query_pairs):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, (query_id, query_text) in enumerate(query_pairs):
        score_vector = np.asarray(bm25.get_scores(tokenize(query_text)), dtype=np.float32)
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": str(query_id),
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_pairs) - 1:
            print(f"  [BM25+] processed {row_index + 1:,}/{len(query_pairs):,} queries")
    return results


def run_embedding_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Run dense semantic retrieval with Sentence-Transformer embeddings."""
    artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame)
    model = artifacts["model"]
    doc_embeddings = artifacts["doc_embeddings"]
    doc_ids = artifacts["doc_ids"]
    query_embeddings = _load_or_encode_embeddings(
        queries_frame,
        kind=embedding_kind,
        model=model,
        model_name=EMBEDDING_MODEL_NAME,
        batch_size=EMBEDDING_BATCH_SIZE,
    )

    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    results: list[RetrievalResult] = []
    total_chunks = (len(query_embeddings) + EMBEDDING_QUERY_CHUNK_SIZE - 1) // EMBEDDING_QUERY_CHUNK_SIZE

    print(
        f"  [Embedding] scoring {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}, chunk_size={EMBEDDING_QUERY_CHUNK_SIZE:,}, "
        f"embedding_cache_key='{embedding_kind}'"
    )

    for chunk_index, start_index in enumerate(range(0, len(query_embeddings), EMBEDDING_QUERY_CHUNK_SIZE), start=1):
        stop_index = start_index + EMBEDDING_QUERY_CHUNK_SIZE
        print(
            f"  [Embedding] chunk {chunk_index:,}/{total_chunks:,}: "
            f"queries {start_index + 1:,}-{min(stop_index, len(query_embeddings)):,}"
        )
        score_block = query_embeddings[start_index:stop_index] @ doc_embeddings.T
        for row_offset, score_vector in enumerate(score_block):
            top_indices = top_k_indices(score_vector, capped_top_k)
            query_id = query_ids[start_index + row_offset]
            results.append(
                {
                    "query_id": query_id,
                    "relevant_docs": doc_ids[top_indices].tolist(),
                }
            )
    return results


MODELS: dict[ModelName, Any] = {
    "tfidf": run_tfidf_search,
    "bm25": run_bm25_search,
    "embedding": run_embedding_search,
}


def run_retrieval(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Dispatch retrieval to the selected method and log the runtime."""
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(MODELS)}")

    print("=" * 88)
    print(f"Starting retrieval: model={model_name}")
    print(
        f"  parameters: top_k={top_k:,}, docs={len(docs_frame):,}, queries={len(queries_frame):,}, "
        f"prepared_artifacts={'yes' if prepared_artifacts is not None else 'no'}, "
        f"embedding_kind='{embedding_kind}'"
    )
    start_time = time.time()
    if model_name == "embedding":
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
            embedding_kind=embedding_kind,
        )
    else:
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
        )
    elapsed_seconds = time.time() - start_time
    print(
        f"Completed retrieval: model={model_name}, results={len(results):,} queries, "
        f"elapsed={elapsed_seconds:.1f}s"
    )
    print("=" * 88)
    return results
validate_pipeline_settings(document_count=len(docs_df))
RETRIEVAL_TOP_K = SUBMIT_TOP_K
print(f"Submission top_k : {SUBMIT_TOP_K:,}")
print(f"Retrieval top_k  : {RETRIEVAL_TOP_K:,}")


## Evaluation Logic

Offline evaluation now mirrors an honest development workflow:

- **Validation split**: used to compare retrieval settings and select the final configuration
- **Hold-out test split**: used once, after validation selection, for a cleaner offline estimate
- **Production test queries**: transformed and scored with the same fitted artifacts, but never used to fit or tune anything

The notebook reports:
- **Recall@K**: how much of the relevant set is recovered
- **Precision@K**: how much of the returned list is relevant
- **MRR@K**: how early the first relevant document appears
- **Accuracy**: category prediction accuracy from the lightweight query classifier

The combined offline score is the simple average of those four values.


In [ ]:
def load_ground_truth(path: Path) -> dict[str, GroundTruthEntry]:
    """Load the training relevance annotations from `qgts_train.json`."""
    if not path.exists():
        raise FileNotFoundError(f"Ground-truth file not found: {path}")

    with open(path, "r", encoding="utf-8") as handle:
        raw_ground_truth = json.load(handle)

    ground_truth: dict[str, GroundTruthEntry] = {}
    for query_id, info in raw_ground_truth.items():
        relevant_items = info.get("relevant_doc_ids", [])
        ground_truth[str(query_id)] = {
            "relevant_doc_ids": {str(item["doc_id"]) for item in relevant_items},
            "total_relevant_docs": int(info.get("total_relevant_docs", len(relevant_items))),
            "category": info.get("category"),
        }
    return ground_truth



def subset_frame_by_ids(frame: pd.DataFrame, query_ids: Sequence[str]) -> pd.DataFrame:
    """Return rows whose string ids belong to `query_ids` while preserving original order."""
    query_id_set = {str(query_id) for query_id in query_ids}
    return (
        frame.assign(id=frame["id"].astype(str))
        .loc[lambda current: current["id"].isin(query_id_set)]
        .reset_index(drop=True)
    )


def split_ground_truth(
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
) -> dict[str, GroundTruthEntry]:
    """Keep only the ground-truth entries that belong to a query split."""
    query_id_set = {str(query_id) for query_id in query_ids}
    return {
        query_id: ground_truth[query_id]
        for query_id in ground_truth
        if query_id in query_id_set
    }


def stratified_query_train_validation_test_split(
    query_frame: pd.DataFrame,
    train_fraction: float = TRAIN_FRACTION,
    validation_fraction: float = VALIDATION_FRACTION,
    test_fraction: float = TEST_FRACTION,
    random_state: int = SPLIT_RANDOM_SEED,
) -> dict[str, pd.DataFrame]:
    """Create a stratified 60/30/10 split over labeled queries."""
    total = train_fraction + validation_fraction + test_fraction
    if not np.isclose(total, 1.0):
        raise ValueError(
            "Split fractions must sum to 1.0; "
            f"got train={train_fraction}, validation={validation_fraction}, test={test_fraction}."
        )

    require_columns(query_frame, ["id", "category"], "Query split frame")
    split_frame = (
        query_frame.copy()
        .assign(id=lambda current: current["id"].astype(str))
        .assign(category=lambda current: current["category"].astype(str))
        .reset_index(drop=True)
    )

    train_frame, remainder_frame = train_test_split(
        split_frame,
        test_size=validation_fraction + test_fraction,
        random_state=random_state,
        stratify=split_frame["category"],
    )

    remainder_test_fraction = test_fraction / (validation_fraction + test_fraction)
    validation_frame, test_frame = train_test_split(
        remainder_frame,
        test_size=remainder_test_fraction,
        random_state=random_state,
        stratify=remainder_frame["category"],
    )

    return {
        "train": train_frame.reset_index(drop=True),
        "validation": validation_frame.reset_index(drop=True),
        "test": test_frame.reset_index(drop=True),
    }


def format_category_distribution(query_frame: pd.DataFrame) -> str:
    """Format category counts for quick split inspection."""
    counts = query_frame["category"].astype(str).value_counts().sort_index()
    return ", ".join(f"{category}={count}" for category, count in counts.items())


def build_split_summary_table(
    query_splits: dict[str, pd.DataFrame],
    total_queries: int,
) -> pd.DataFrame:
    """Summarize split sizes and category balance in one compact table."""
    rows = []
    for split_name in ("train", "validation", "test"):
        split_frame = query_splits[split_name]
        rows.append(
            {
                "Split": "holdout_test" if split_name == "test" else split_name,
                "Rows": len(split_frame),
                "Share": len(split_frame) / total_queries,
                "Categories": format_category_distribution(split_frame),
            }
        )
    return pd.DataFrame(rows)


def build_classifier_summary_table(
    rows: Sequence[tuple[str, int, float | None]],
) -> pd.DataFrame:
    """Summarize classifier coverage and accuracy by split."""
    return pd.DataFrame(
        [
            {
                "Split": split_name,
                "Rows": row_count,
                "Accuracy": np.nan if accuracy is None else accuracy,
            }
            for split_name, row_count, accuracy in rows
        ]
    )


def recall_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Recall@K across all queries present in the ground truth."""
    recalls: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        total_relevant_docs = ground_truth[query_id]["total_relevant_docs"]
        predicted_doc_ids = item["relevant_docs"][:k]
        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        recall_value = hits / total_relevant_docs if total_relevant_docs > 0 else 0.0
        recalls.append(recall_value)

    return float(np.mean(recalls)) if recalls else 0.0


def precision_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Precision@K across all queries present in the ground truth."""
    precisions: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        predicted_doc_ids = item["relevant_docs"][:k]
        if not predicted_doc_ids:
            precisions.append(0.0)
            continue

        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        precisions.append(hits / len(predicted_doc_ids))

    return float(np.mean(precisions)) if precisions else 0.0


def mrr_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean reciprocal rank at K."""
    reciprocal_ranks: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(item["relevant_docs"][:k], start=1):
            if doc_id in relevant_doc_ids:
                reciprocal_rank = 1.0 / rank
                break
        reciprocal_ranks.append(reciprocal_rank)

    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0


def compute_category_accuracy(
    ground_truth: dict[str, GroundTruthEntry],
    predicted_categories: dict[str, str] | None,
    default_if_missing: float = 0.0,
) -> float:
    """Compute query category accuracy when category predictions are available."""
    if predicted_categories is None:
        return float(default_if_missing)

    try:
        from sklearn.metrics import accuracy_score
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    y_true: list[str] = []
    y_pred: list[str] = []
    for query_id, info in ground_truth.items():
        true_category = info.get("category")
        predicted_category = predicted_categories.get(str(query_id))
        if true_category is None or predicted_category is None:
            continue
        y_true.append(str(true_category))
        y_pred.append(str(predicted_category))

    if not y_true:
        return float(default_if_missing)
    return float(accuracy_score(y_true, y_pred))


def leaderboard_score(
    results: list[RetrievalResult],
    ground_truth: dict[str, GroundTruthEntry],
    k: int,
    predicted_categories: dict[str, str] | None = None,
    accuracy_value: float | None = None,
) -> dict[str, float]:
    """Compute the combined offline leaderboard-style score."""
    recall_value = recall_at_k(results, ground_truth, k=k)
    precision_value = precision_at_k(results, ground_truth, k=k)
    mrr_value = mrr_at_k(results, ground_truth, k=k)
    category_accuracy = (
        float(accuracy_value)
        if accuracy_value is not None
        else compute_category_accuracy(ground_truth, predicted_categories)
    )
    combined_score = 0.25 * (recall_value + precision_value + mrr_value + category_accuracy)
    return {
        "Recall": recall_value,
        "Precision": precision_value,
        "MRR": mrr_value,
        "Accuracy": category_accuracy,
        "LeaderboardScore": combined_score,
    }


def predict_category_map(query_frame: pd.DataFrame, classifier_artifacts: dict[str, Any]) -> dict[str, str]:
    """Predict one category label per query using the cached classifier."""
    query_vectors = classifier_artifacts["vectorizer"].transform(query_frame["content"])
    predictions = classifier_artifacts["classifier"].predict(query_vectors)
    return {
        str(query_id): str(prediction)
        for query_id, prediction in zip(query_frame["id"].astype(str), predictions)
    }


def build_doc_category_map(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build a lookup from document id to document category."""
    require_columns(docs_frame, ["id", "category"], "Documents frame")
    return (
        docs_frame[["id", "category"]]
        .assign(id=lambda frame: frame["id"].astype(str))
        .set_index("id")["category"]
        .to_dict()
    )


def build_text_map(frame: pd.DataFrame, id_column: str = "id", text_column: str = "content") -> dict[str, str]:
    """Build a string-id to text lookup from a dataframe."""
    require_columns(frame, [id_column, text_column], f"Frame[{id_column}, {text_column}]")
    return {
        str(row_id): str(text)
        for row_id, text in frame[[id_column, text_column]].itertuples(index=False, name=None)
    }


def sample_random_negative_doc_ids(
    all_doc_ids: Sequence[str],
    excluded_doc_ids: set[str],
    sample_size: int,
    rng: Any,
) -> list[str]:
    """Sample random negative doc ids while excluding known relevant ids."""
    if sample_size <= 0:
        return []
    if not all_doc_ids:
        return []

    negatives: list[str] = []
    max_attempts = max(100, sample_size * 50)
    attempts = 0
    while len(negatives) < sample_size and attempts < max_attempts:
        candidate = str(all_doc_ids[rng.randrange(len(all_doc_ids))])
        attempts += 1
        if candidate in excluded_doc_ids or candidate in negatives:
            continue
        negatives.append(candidate)
    return negatives


def mine_hard_negative_doc_ids(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    top_k: int = CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
    prepared_artifacts: dict[str, Any] | None = None,
) -> dict[str, list[str]]:
    """Mine hard negatives from top embedding hits that are not relevant."""
    if top_k <= 0:
        return {}

    query_id_set = {str(query_id) for query_id in query_ids}
    mining_queries_frame = (
        train_queries_frame.assign(id=train_queries_frame["id"].astype(str))
        .loc[lambda frame: frame["id"].isin(query_id_set), ["id", "content"]]
        .reset_index(drop=True)
    )
    if mining_queries_frame.empty:
        return {}

    embedding_artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame)
    retrieval_results = run_retrieval(
        model_name="embedding",
        docs_frame=docs_frame,
        queries_frame=mining_queries_frame,
        top_k=top_k,
        prepared_artifacts=embedding_artifacts,
        embedding_kind="queries_train_hardneg",
    )

    hard_negative_doc_ids_by_query: dict[str, list[str]] = {}
    for item in retrieval_results:
        query_id = str(item["query_id"])
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"] if query_id in ground_truth else set()
        negatives: list[str] = []
        seen_doc_ids: set[str] = set()
        for doc_id in item["relevant_docs"]:
            candidate_doc_id = str(doc_id)
            if candidate_doc_id in relevant_doc_ids or candidate_doc_id in seen_doc_ids:
                continue
            negatives.append(candidate_doc_id)
            seen_doc_ids.add(candidate_doc_id)
        hard_negative_doc_ids_by_query[query_id] = negatives

    counts = [len(doc_ids) for doc_ids in hard_negative_doc_ids_by_query.values()]
    if counts:
        print(
            f"  [HardNegatives] mined for {len(counts):,} queries "
            f"(per-query min/mean/max={min(counts):,}/{float(np.mean(counts)):.1f}/{max(counts):,}, top_k={top_k:,})"
        )
    return hard_negative_doc_ids_by_query


def build_cross_encoder_training_examples(
    query_text_map: dict[str, str],
    doc_text_map: dict[str, str],
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    max_positives_per_query: int = CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
    negatives_per_positive: int = CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
    hard_negative_doc_ids_by_query: dict[str, list[str]] | None = None,
    seed: int = CROSS_ENCODER_RANDOM_SEED,
) -> list[Any]:
    """Build binary (query, doc) examples using positives plus mined hard negatives."""
    try:
        from sentence_transformers import InputExample
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    import random

    rng = random.Random(seed)
    all_doc_ids = list(doc_text_map.keys())
    examples: list[Any] = []
    hard_negative_count = 0
    random_negative_count = 0

    for query_id in query_ids:
        query_id_str = str(query_id)
        query_text = query_text_map.get(query_id_str)
        if query_text is None:
            continue
        if query_id_str not in ground_truth:
            continue

        relevant_doc_ids = [
            str(doc_id)
            for doc_id in ground_truth[query_id_str]["relevant_doc_ids"]
            if str(doc_id) in doc_text_map
        ]
        if not relevant_doc_ids:
            continue

        rng.shuffle(relevant_doc_ids)
        selected_positive_doc_ids = relevant_doc_ids[:max_positives_per_query]
        relevant_doc_id_set = set(relevant_doc_ids)
        hard_negative_pool = [
            doc_id
            for doc_id in (hard_negative_doc_ids_by_query or {}).get(query_id_str, [])
            if doc_id in doc_text_map and doc_id not in relevant_doc_id_set
        ]

        for positive_doc_id in selected_positive_doc_ids:
            examples.append(InputExample(texts=[query_text, doc_text_map[positive_doc_id]], label=1.0))
            negative_doc_ids: list[str] = []

            if hard_negative_pool:
                hard_take = min(negatives_per_positive, len(hard_negative_pool))
                negative_doc_ids.extend(rng.sample(hard_negative_pool, hard_take))
                hard_negative_count += hard_take

            if len(negative_doc_ids) < negatives_per_positive:
                random_needed = negatives_per_positive - len(negative_doc_ids)
                extra_random_negatives = sample_random_negative_doc_ids(
                    all_doc_ids=all_doc_ids,
                    excluded_doc_ids=relevant_doc_id_set | set(negative_doc_ids),
                    sample_size=random_needed,
                    rng=rng,
                )
                negative_doc_ids.extend(extra_random_negatives)
                random_negative_count += len(extra_random_negatives)

            for negative_doc_id in negative_doc_ids:
                examples.append(InputExample(texts=[query_text, doc_text_map[negative_doc_id]], label=0.0))

    print(
        f"Cross-encoder pairs: total={len(examples):,}, "
        f"hard_negatives={hard_negative_count:,}, random_negatives={random_negative_count:,}"
    )
    return examples


def build_or_load_cross_encoder(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
) -> Any:
    """Train or load a cached cross-encoder reranker fit only on the provided training split."""
    try:
        from sentence_transformers import CrossEncoder
        from torch.utils.data import DataLoader
        import torch
    except ImportError as exc:
        raise ImportError(
            "Missing dependencies for cross-encoder training. Install `%pip install sentence-transformers torch`."
        ) from exc

    query_text_map = build_text_map(train_queries_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")

    candidate_query_ids = [
        query_id
        for query_id in train_queries_frame["id"].astype(str).tolist()
        if query_id in ground_truth and query_id in query_text_map
    ]
    if CROSS_ENCODER_TRAIN_QUERY_LIMIT > 0:
        candidate_query_ids = candidate_query_ids[:CROSS_ENCODER_TRAIN_QUERY_LIMIT]
    if not candidate_query_ids:
        raise ValueError("No training queries available for cross-encoder training.")

    train_signature = _hash_payload(
        {
            "query_signature": _dataframe_fingerprint(train_queries_frame, ["id", "content"]),
            "doc_signature": _dataframe_fingerprint(docs_frame, ["id", "content"]),
            "query_count": len(candidate_query_ids),
            "model": CROSS_ENCODER_MODEL_NAME,
            "epochs": CROSS_ENCODER_EPOCHS,
            "batch_size": CROSS_ENCODER_BATCH_SIZE,
            "max_length": CROSS_ENCODER_MAX_LENGTH,
            "max_pos_per_query": CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
            "neg_per_pos": CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
            "hard_neg_top_k": CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
            "hard_neg_model": "embedding",
            "hard_neg_embedding_model": EMBEDDING_MODEL_NAME,
            "seed": CROSS_ENCODER_RANDOM_SEED,
        }
    )
    model_dir = CROSS_ENCODER_CACHE_DIR / f"{_safe_component(CROSS_ENCODER_MODEL_NAME)}_{train_signature}"
    cache_marker = model_dir / "config.json"
    memory_key = f"cross_encoder::{model_dir.resolve()}"

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]

    if ENABLE_CROSS_ENCODER_CACHE and cache_marker.exists():
        print(f"Loading cross-encoder from cache: {model_dir.name}")
        try:
            cached_model = CrossEncoder(str(model_dir), max_length=CROSS_ENCODER_MAX_LENGTH)
            if CROSS_ENCODER_FP16 and torch.cuda.is_available():
                cached_model.model.half()
            _OBJECT_MEMORY_CACHE[memory_key] = cached_model
            return cached_model
        except Exception as exc:
            print(f"Cross-encoder cache load failed, retraining: {exc}")
    elif ENABLE_CROSS_ENCODER_CACHE and model_dir.exists():
        print(f"Cross-encoder cache directory exists but is incomplete: {model_dir}")

    embedding_artifacts = prepare_retriever("embedding", docs_frame)
    hard_negative_doc_ids_by_query = mine_hard_negative_doc_ids(
        train_queries_frame=train_queries_frame,
        docs_frame=docs_frame,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        top_k=CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
        prepared_artifacts=embedding_artifacts,
    )

    training_examples = build_cross_encoder_training_examples(
        query_text_map=query_text_map,
        doc_text_map=doc_text_map,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        max_positives_per_query=CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
        negatives_per_positive=CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
        hard_negative_doc_ids_by_query=hard_negative_doc_ids_by_query,
        seed=CROSS_ENCODER_RANDOM_SEED,
    )
    if not training_examples:
        raise ValueError("Cross-encoder training set is empty after preprocessing.")

    print(
        f"Training cross-encoder on {len(training_examples):,} pairs "
        f"from {len(candidate_query_ids):,} queries"
    )
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=CROSS_ENCODER_MAX_LENGTH)
    train_loader = DataLoader(training_examples, shuffle=True, batch_size=CROSS_ENCODER_BATCH_SIZE)
    warmup_steps = max(1, int(len(train_loader) * CROSS_ENCODER_EPOCHS * 0.1))
    if ENABLE_CROSS_ENCODER_CACHE:
        model_dir.mkdir(parents=True, exist_ok=True)
    output_path = str(model_dir) if ENABLE_CROSS_ENCODER_CACHE else None
    cross_encoder.fit(
        train_dataloader=train_loader,
        epochs=CROSS_ENCODER_EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        output_path=output_path,
    )

    if ENABLE_CROSS_ENCODER_CACHE:
        print(f"Saved cross-encoder to cache: {model_dir.name}")
        cross_encoder.save(str(model_dir))
        cached_model = CrossEncoder(str(model_dir), max_length=CROSS_ENCODER_MAX_LENGTH)
        if CROSS_ENCODER_FP16 and torch.cuda.is_available():
            cached_model.model.half()
        _OBJECT_MEMORY_CACHE[memory_key] = cached_model
        return cached_model

    if CROSS_ENCODER_FP16 and torch.cuda.is_available():
        cross_encoder.model.half()
    _OBJECT_MEMORY_CACHE[memory_key] = cross_encoder
    return cross_encoder
def compute_category_boost(
    query_id: str,
    doc_ids: list[str],
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    category_bonus: float
) -> np.ndarray:
    """
    Computes a flat scalar bonus for documents matching the predicted query category.
    This replaces the strict filter, preserving true positives that the classifier missed.
    """
    boosts = np.zeros(len(doc_ids), dtype=np.float32)
    predicted_query_cat = query_category_map.get(query_id)

    if predicted_query_cat is None or category_bonus == 0.0:
        return boosts

    target_category = str(predicted_query_cat)

    for i, doc_id in enumerate(doc_ids):
        raw_doc_cat = doc_category_map.get(doc_id)
        doc_category = "unknown" if raw_doc_cat is None or pd.isna(raw_doc_cat) else str(raw_doc_cat)

        if doc_category == target_category:
            boosts[i] = category_bonus

    return boosts

def rerank_results_with_cross_encoder(
    results: list[RetrievalResult],
    query_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    cross_encoder: Any,
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    rerank_top_m: int = 100,
    category_bonus: float = 2.0,
) -> list[RetrievalResult]:
    """Rerank the top-m candidates with the cross-encoder plus an optional category bonus."""
    if rerank_top_m <= 0:
        raise ValueError("rerank_top_m must be positive.")

    query_text_map = build_text_map(query_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    # --- Pass 1: collect all pairs and per-query metadata ---
    all_pairs: list[list[str]] = []
    query_meta: list[dict] = []

    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]

        if query_id not in query_text_map:
            query_meta.append({"query_id": query_id, "skip": True, "doc_ids": doc_ids})
            continue

        head_doc_ids = doc_ids[:rerank_top_m]
        tail_doc_ids = doc_ids[rerank_top_m:]
        scored_doc_ids = [d for d in head_doc_ids if d in doc_text_map]
        missing_head_doc_ids = [d for d in head_doc_ids if d not in doc_text_map]

        if len(scored_doc_ids) <= 1:
            query_meta.append({"query_id": query_id, "skip": True, "doc_ids": doc_ids})
            continue

        query_text = query_text_map[query_id]
        pairs = [[query_text, doc_text_map[d]] for d in scored_doc_ids]
        start = len(all_pairs)
        all_pairs.extend(pairs)
        query_meta.append({
            "query_id": query_id,
            "skip": False,
            "scored_doc_ids": scored_doc_ids,
            "missing_head_doc_ids": missing_head_doc_ids,
            "tail_doc_ids": tail_doc_ids,
            "slice": (start, start + len(pairs)),
        })

    # --- Single global predict() call across all queries ---
    if all_pairs:
        all_scores = np.asarray(
            cross_encoder.predict(
                all_pairs,
                batch_size=CROSS_ENCODER_INFER_BATCH_SIZE,
                show_progress_bar=False,
            ),
            dtype=np.float32,
        )
    else:
        all_scores = np.array([], dtype=np.float32)

    # --- Pass 2: reconstruct reranked results ---
    reranked_results: list[RetrievalResult] = []
    reranked_query_count = 0

    for meta in query_meta:
        if meta["skip"]:
            reranked_results.append({"query_id": meta["query_id"], "relevant_docs": meta["doc_ids"]})
            continue

        start, end = meta["slice"]
        ce_scores = all_scores[start:end]
        scored_doc_ids = meta["scored_doc_ids"]
        boosts = compute_category_boost(
            query_id=meta["query_id"],
            doc_ids=scored_doc_ids,
            query_category_map=query_category_map,
            doc_category_map=doc_category_map,
            category_bonus=category_bonus,
        )
        final_scores = ce_scores + boosts
        ranked_indices = np.argsort(final_scores)[::-1]
        reranked_scored_doc_ids = [scored_doc_ids[i] for i in ranked_indices]
        reranked_doc_ids = reranked_scored_doc_ids + meta["missing_head_doc_ids"] + meta["tail_doc_ids"]
        reranked_results.append({"query_id": meta["query_id"], "relevant_docs": reranked_doc_ids})
        reranked_query_count += 1

    print(
        f"  [CrossEncoder] reranked {reranked_query_count:,}/{len(results):,} queries "
        f"(top_m={rerank_top_m:,}, category_bonus={category_bonus:.2f}, "
        f"total_pairs={len(all_pairs):,})"
    )
    return reranked_results


def ensure_prepared_retriever(
    prepared_retrievers: dict[ModelName, dict[str, Any]],
    model_name: ModelName,
    docs_frame: pd.DataFrame,
) -> dict[str, Any]:
    """Build one retriever lazily when it is not already cached in memory."""
    artifacts = prepared_retrievers.get(model_name)
    if artifacts is None:
        print(f"Preparing artifacts for {model_name}...")
        artifacts = prepare_retriever(model_name, docs_frame)
        prepared_retrievers[model_name] = artifacts
    return artifacts


def run_ranked_results(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any],
    embedding_kind: str,
    cross_encoder: Any | None = None,
    query_category_map: dict[str, str] | None = None,
    doc_category_map: dict[str, Any] | None = None,
    rerank_top_m: int = CROSS_ENCODER_RERANK_TOP_M,
    category_bonus: float = 0.0,
) -> list[RetrievalResult]:
    """Run retrieval and optional reranking for one query frame."""
    results = run_retrieval(
        model_name=model_name,
        docs_frame=docs_frame,
        queries_frame=queries_frame,
        top_k=top_k,
        prepared_artifacts=prepared_artifacts,
        embedding_kind=embedding_kind,
    )
    if cross_encoder is None:
        return results
    if query_category_map is None or doc_category_map is None:
        raise ValueError("query_category_map and doc_category_map are required when reranking.")
    return rerank_results_with_cross_encoder(
        results=results,
        query_frame=queries_frame,
        docs_frame=docs_frame,
        cross_encoder=cross_encoder,
        query_category_map=query_category_map,
        doc_category_map=doc_category_map,
        rerank_top_m=rerank_top_m,
        category_bonus=category_bonus,
    )


def write_kaggle_submission(
    results: list[RetrievalResult],
    sample_csv_path: Path,
    output_csv_path: Path,
    category_predictions: dict[str, str] | None = None,
) -> None:
    """Write predictions in the exact Kaggle submission format."""
    prediction_map = {
        str(item["query_id"]): [str(doc_id) for doc_id in item["relevant_docs"]]
        for item in results
    }

    with open(sample_csv_path, "r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)

    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError("Invalid sample submission format.")

    query_id_column = fieldnames[0]
    prediction_column = fieldnames[1]
    category_column = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            query_id = str(row[query_id_column])
            if query_id not in prediction_map:
                raise ValueError(f"Missing retrieval prediction for query_id={query_id}")

            output_row = {
                query_id_column: query_id,
                prediction_column: json.dumps(prediction_map[query_id]),
            }
            if category_column is not None:
                if category_predictions is None:
                    output_row[category_column] = row.get(category_column, "?") or "?"
                else:
                    if query_id not in category_predictions:
                        raise ValueError(f"Missing category prediction for query_id={query_id}")
                    output_row[category_column] = str(category_predictions[query_id])
            writer.writerow(output_row)


ground_truth = load_ground_truth(ground_truth_path)
print(f"Ground-truth queries: {len(ground_truth):,}")


## Interpreting Recall, Precision, MRR, and Accuracy

The four reported metrics answer different questions:

- **Recall@K**: "Did we recover most of the relevant documents at all?"
- **Precision@K**: "How much noise is in the returned list?"
- **MRR@K**: "How early does the first useful document appear?"
- **Accuracy**: "Did the classifier predict the correct category label?"

Interpretation tips:
- If validation recall is low, the first-stage retriever is failing to surface relevant candidates.
- If recall is high but MRR is low, the right documents are present but ranked too deep.
- If validation is strong but hold-out test falls sharply, the configuration is probably overfit to validation.
- If accuracy stays flat while retrieval metrics move, the change is coming from the retriever or reranker rather than the classifier.


## Cross-Encoder Reranking

A cross-encoder is a **second-stage ranker**: instead of encoding query and document independently, it scores each `(query, document)` pair jointly.

Why this helps MRR:
- first-stage retrieval focuses on recall and usually returns a noisy candidate set
- cross-encoders are slower but better at fine-grained ordering near the top
- better ordering near the top is exactly what MRR rewards

In this notebook the reranker is also leakage-safe:
- train only on the 60% split-train queries and their relevance labels
- rerank only the top segment (`top_m`) per query to keep runtime bounded
- reuse the same trained reranker on validation, hold-out test, and production queries without refitting


## Experiments

The experiment section now stays close to the final workflow:

- create a stratified `60/30/10` split
- fit the query category classifier only on split-train queries
- train or load the cross-encoder only on split-train queries
- prepare retriever artifacts once on the document corpus
- compare settings on validation only
- run one hold-out test evaluation for the selected validation winner
- reuse the same fitted artifacts for Kaggle production predictions


In [ ]:
ground_truth = load_ground_truth(ground_truth_path)
print(f"Ground-truth queries: {len(ground_truth):,}")

query_splits = stratified_query_train_validation_test_split(train_queries_df)
split_train_queries_df = query_splits["train"]
validation_queries_df = query_splits["validation"]
holdout_test_queries_df = query_splits["test"]

split_train_queries_classifier_df = subset_frame_by_ids(
    train_queries_classifier_df,
    split_train_queries_df["id"],
)
validation_queries_classifier_df = subset_frame_by_ids(
    train_queries_classifier_df,
    validation_queries_df["id"],
)
holdout_test_queries_classifier_df = subset_frame_by_ids(
    train_queries_classifier_df,
    holdout_test_queries_df["id"],
)

train_ground_truth = split_ground_truth(ground_truth, split_train_queries_df["id"])
validation_ground_truth = split_ground_truth(ground_truth, validation_queries_df["id"])
holdout_test_ground_truth = split_ground_truth(ground_truth, holdout_test_queries_df["id"])

split_summary_df = build_split_summary_table(query_splits, len(train_queries_df))
print("\nQuery split summary:")
print(split_summary_df.to_string(index=False))

category_classifier = build_or_load_category_classifier(
    split_train_queries_classifier_df[["id", "content", "category"]].copy()
)
train_query_category_map = predict_category_map(split_train_queries_classifier_df, category_classifier)
validation_query_category_map = predict_category_map(validation_queries_classifier_df, category_classifier)
holdout_test_query_category_map = predict_category_map(
    holdout_test_queries_classifier_df,
    category_classifier,
)
production_query_category_map = predict_category_map(test_queries_classifier_df, category_classifier)

train_classifier_accuracy = compute_category_accuracy(train_ground_truth, train_query_category_map)
validation_classifier_accuracy = compute_category_accuracy(
    validation_ground_truth,
    validation_query_category_map,
)
holdout_test_classifier_accuracy = compute_category_accuracy(
    holdout_test_ground_truth,
    holdout_test_query_category_map,
)

classifier_summary_df = build_classifier_summary_table(
    [
        ("train", len(split_train_queries_classifier_df), train_classifier_accuracy),
        ("validation", len(validation_queries_classifier_df), validation_classifier_accuracy),
        ("holdout_test", len(holdout_test_queries_classifier_df), holdout_test_classifier_accuracy),
        ("production", len(test_queries_classifier_df), None),
    ]
)
print("\nCategory classifier summary (fit on split-train only):")
print(classifier_summary_df.to_string(index=False))

doc_category_map = build_doc_category_map(docs_df)
cross_encoder_reranker = None
if ENABLE_CROSS_ENCODER_RERANK:
    cross_encoder_reranker = build_or_load_cross_encoder(
        train_queries_frame=split_train_queries_df,
        docs_frame=docs_df,
        ground_truth=train_ground_truth,
    )

prepared_retrievers: dict[ModelName, dict[str, Any]] = {}
for model_name in set(EVALUATION_MODELS) | {FINAL_MODEL}:
    ensure_prepared_retriever(prepared_retrievers, model_name, docs_df)

category_bonus = CATEGORY_MATCH_BONUS if ENABLE_CATEGORY_BOOST else 0.0
max_eval_top_k = max(EVALUATION_TOP_KS)
validation_rows: list[dict[str, Any]] = []

for model_name in EVALUATION_MODELS:
    validation_results = run_ranked_results(
        model_name=model_name,
        docs_frame=docs_df,
        queries_frame=validation_queries_df,
        top_k=max_eval_top_k,
        prepared_artifacts=prepared_retrievers[model_name],
        embedding_kind="queries_validation",
        cross_encoder=cross_encoder_reranker,
        query_category_map=validation_query_category_map,
        doc_category_map=doc_category_map,
        rerank_top_m=CROSS_ENCODER_RERANK_TOP_M,
        category_bonus=category_bonus,
    )

    for top_k in EVALUATION_TOP_KS:
        metrics = leaderboard_score(
            truncate_results(validation_results, top_k),
            validation_ground_truth,
            k=top_k,
            accuracy_value=validation_classifier_accuracy,
        )
        validation_rows.append(
            {
                "Evaluation": "validation",
                "Model": model_name,
                "TopK": top_k,
                **metrics,
            }
        )
        print(
            f"[validation] {model_name:10s} top_k={top_k:>6,}  "
            f"Recall={metrics['Recall']:.5f}  "
            f"Precision={metrics['Precision']:.5f}  "
            f"MRR={metrics['MRR']:.5f}  "
            f"Accuracy={metrics['Accuracy']:.5f}  "
            f"Score={metrics['LeaderboardScore']:.5f}"
        )

validation_summary_df = (
    pd.DataFrame(validation_rows)
    .sort_values(["LeaderboardScore", "TopK"], ascending=[False, False])
    .reset_index(drop=True)
)
if validation_summary_df.empty:
    raise ValueError("No validation rows were produced.")

best_validation_config = validation_summary_df.iloc[0]
best_model_name = str(best_validation_config["Model"])
best_top_k = int(best_validation_config["TopK"])

holdout_test_results = run_ranked_results(
    model_name=best_model_name,
    docs_frame=docs_df,
    queries_frame=holdout_test_queries_df,
    top_k=best_top_k,
    prepared_artifacts=prepared_retrievers[best_model_name],
    embedding_kind="queries_holdout_test",
    cross_encoder=cross_encoder_reranker,
    query_category_map=holdout_test_query_category_map,
    doc_category_map=doc_category_map,
    rerank_top_m=CROSS_ENCODER_RERANK_TOP_M,
    category_bonus=category_bonus,
)

holdout_test_metrics = leaderboard_score(
    holdout_test_results,
    holdout_test_ground_truth,
    k=best_top_k,
    accuracy_value=holdout_test_classifier_accuracy,
)
holdout_test_summary_df = pd.DataFrame(
    [
        {
            "Evaluation": "holdout_test",
            "Model": best_model_name,
            "TopK": best_top_k,
            **holdout_test_metrics,
        }
    ]
)

print("\nBest validation configuration:")
print(best_validation_config.to_string())
print("\nHold-out test score for the selected validation winner:")
print(holdout_test_summary_df.to_string(index=False))

evaluation_summary_df = pd.concat(
    [validation_summary_df, holdout_test_summary_df],
    ignore_index=True,
)
evaluation_summary_df


## Generate Test Submission

Kaggle test queries are treated as future production inputs: the split-train classifier vectorizer and the split-train cross-encoder are reused as-is, with no refitting during submission generation.


In [ ]:
final_model_artifacts = ensure_prepared_retriever(prepared_retrievers, FINAL_MODEL, docs_df)

print("Generating production predictions with train-fitted artifacts only.")
production_results = run_ranked_results(
    model_name=FINAL_MODEL,
    docs_frame=docs_df,
    queries_frame=test_queries_df,
    top_k=RETRIEVAL_TOP_K,
    prepared_artifacts=final_model_artifacts,
    embedding_kind="queries_production",
    cross_encoder=cross_encoder_reranker if ENABLE_CROSS_ENCODER_RERANK else None,
    query_category_map=production_query_category_map,
    doc_category_map=doc_category_map,
    rerank_top_m=CROSS_ENCODER_RERANK_TOP_M,
    category_bonus=category_bonus,
)

write_kaggle_submission(
    production_results,
    sample_csv_path=sample_submission_path,
    output_csv_path=PATHS.output_path,
    category_predictions=production_query_category_map,
)

submission_preview = pd.read_csv(PATHS.output_path)
print(f"Saved submission to: {PATHS.output_path.resolve()}")
print(f"Rows: {len(submission_preview):,}")
submission_preview.head()


## Conclusions

The notebook now stays focused on the final pipeline: split-aware evaluation, train-only fitted preprocessing, cached retrieval artifacts, and production submission generation. The core retrieval setup is unchanged, but the notebook is shorter and easier to scan.
